In [42]:
!pip install torch torchvision



In [43]:
import torch
import torchvision
print(torch.__version__)
print(torchvision.__version__)

2.10.0+cpu
0.25.0+cpu


In [44]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torchvision.models import ResNet18_Weights
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

# 하이퍼파라미터 설정
CFG = {
    'IMG_SIZE': 224,
    'EPOCHS': 12,
    'LEARNING_RATE': 3e-4,
    'MIN_LR': 1e-6,
    'BATCH_SIZE': 32,
    'WEIGHT_DECAY': 1e-4,
    'SEED': 42,
    'SEEDS': [42, 3407, 777],
    'NUM_WORKERS': 0,
    'TTA_HFLIP': True
}

def seed_everything(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(CFG['SEED'])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ID prefix 기준 폴더 매핑
DATA_ROOTS = {
    'TRAIN': Path('data/train'),
    'DEV': Path('data/dev'),
    'TEST': Path('data/test')
}



In [45]:
# 1. 데이터 로드
train_df = pd.read_csv('data/train.csv')
val_df = pd.read_csv('data/dev.csv')

print(f"학습 데이터 개수: {len(train_df)}")
print(f"검증 데이터 개수: {len(val_df)}")



학습 데이터 개수: 1000
검증 데이터 개수: 100


In [46]:
class MultiViewDataset(Dataset):
    def __init__(self, df, transform=None, is_test=False):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.is_test = is_test
        self.label_map = {'stable': 0, 'unstable': 1}

    def __len__(self):
        return len(self.df)

    def _resolve_folder(self, sample_id: str) -> Path:
        prefix = sample_id.split('_')[0]
        if prefix not in DATA_ROOTS:
            raise ValueError(f"Unknown id prefix: {sample_id}")
        return DATA_ROOTS[prefix] / sample_id

    def __getitem__(self, idx):
        sample_id = str(self.df.iloc[idx]['id'])
        folder_path = self._resolve_folder(sample_id)

        # 2개 뷰 로드
        views = []
        for name in ["front", "top"]:
            img_path = folder_path / f"{name}.png"
            image = Image.open(img_path).convert("RGB")
            if self.transform:
                image = self.transform(image)
            views.append(image)

        # 테스트(추론) 모드일 경우 이미지 리스트만 반환
        if self.is_test:
            return views

        # 학습/검증 모드일 경우 라벨 함께 반환
        label = self.label_map[self.df.iloc[idx]['label']]
        return views, label



In [47]:
train_transform = transforms.Compose([
    transforms.Resize((CFG['IMG_SIZE'], CFG['IMG_SIZE'])),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomApply([
        transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.2, hue=0.05)
    ], p=0.8),
    transforms.RandomApply([
        transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5))
    ], p=0.3),
    transforms.RandomAffine(
        degrees=8,
        translate=(0.05, 0.05),
        scale=(0.92, 1.08),
        shear=5
    ),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((CFG['IMG_SIZE'], CFG['IMG_SIZE'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 1. 학습/검증 세트 준비 (is_test=False 설정)
train_dataset = MultiViewDataset(train_df, train_transform, is_test=False)
val_dataset = MultiViewDataset(val_df, test_transform, is_test=False)

train_loader = DataLoader(
    train_dataset,
    batch_size=CFG['BATCH_SIZE'],
    shuffle=True,
    num_workers=CFG['NUM_WORKERS']
)
val_loader = DataLoader(
    val_dataset,
    batch_size=CFG['BATCH_SIZE'],
    shuffle=False,
    num_workers=CFG['NUM_WORKERS']
)

# 2. 테스트 세트 준비 (is_test=True 설정)
test_df = pd.read_csv('data/sample_submission.csv')
test_dataset = MultiViewDataset(test_df, test_transform, is_test=True)
test_loader = DataLoader(
    test_dataset,
    batch_size=CFG['BATCH_SIZE'],
    shuffle=False,
    num_workers=CFG['NUM_WORKERS']
)



### 4.Multi-View

In [48]:
class MultiViewResNet(nn.Module):
    def __init__(self, num_classes=1):
        super(MultiViewResNet, self).__init__()
        self.backbone = models.resnet18(weights=ResNet18_Weights.DEFAULT)
        self.feature_extractor = nn.Sequential(*list(self.backbone.children())[:-1])
        
        self.classifier = nn.Sequential(
            nn.Linear(512 * 2, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )

    def forward(self, views):
        # views: [batch, 3, 224, 224] * 2 images
        f1 = self.feature_extractor(views[0]).view(views[0].size(0), -1)
        f2 = self.feature_extractor(views[1]).view(views[1].size(0), -1)
        
        combined = torch.cat((f1, f2), dim=1)
        return self.classifier(combined)

In [49]:
def LOGLOSS(true, pred, eps=1e-15):
    pred = np.clip(pred, eps, 1 - eps)
    pred = pred / np.sum(pred, axis=1).reshape(-1, 1)
    loss = -np.sum(true * np.log(pred), axis=1)
    return np.mean(loss)

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    train_loss = 0
    for views, labels in tqdm(loader, desc="Training"):
        views = [v.to(device) for v in views]
        labels = labels.to(device).float()

        optimizer.zero_grad()
        outputs = model(views).view(-1)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
    return train_loss / len(loader)

def validate(model, loader, criterion, device):
    model.eval()
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for views, labels in tqdm(loader, desc="Validation"):
            views = [v.to(device) for v in views]
            labels = labels.to(device).float()

            outputs = model(views).view(-1)
            probs = torch.sigmoid(outputs)

            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    all_probs = np.array(all_probs, dtype=np.float64)
    all_labels = np.array(all_labels, dtype=np.float64)

    # 대회 컬럼 순서: [unstable_prob, stable_prob]
    pred_2col = np.stack([all_probs, 1.0 - all_probs], axis=1)
    true_2col = np.stack([all_labels, 1.0 - all_labels], axis=1)

    logloss_score = LOGLOSS(true_2col, pred_2col)
    acc_score = np.mean((all_probs > 0.5) == all_labels)

    return logloss_score, acc_score



In [50]:
criterion = nn.BCEWithLogitsLoss()
best_ckpts = []
seed_val_scores = {}

for seed in CFG['SEEDS']:
    print(f"\n===== Seed {seed} Training Start =====")
    seed_everything(seed)

    model = MultiViewResNet().to(device)
    optimizer = optim.AdamW(
        model.parameters(),
        lr=CFG['LEARNING_RATE'],
        weight_decay=CFG['WEIGHT_DECAY']
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=CFG['EPOCHS'],
        eta_min=CFG['MIN_LR']
    )

    best_logloss = float('inf')
    best_path = f'best_model_seed{seed}.pt'

    for epoch in range(1, CFG['EPOCHS'] + 1):
        avg_train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_logloss, val_acc = validate(model, val_loader, criterion, device)

        scheduler.step()
        current_lr = optimizer.param_groups[0]['lr']

        if val_logloss < best_logloss:
            best_logloss = val_logloss
            torch.save(model.state_dict(), best_path)

        print(f"Seed [{seed}] Epoch [{epoch}]")
        print(f"  - LR: {current_lr:.7f}")
        print(f"  - Train Loss: {avg_train_loss:.4f}")
        print(f"  - Val Log-Loss: {val_logloss:.6f} | Val Acc: {val_acc:.4f}")

    best_ckpts.append(best_path)
    seed_val_scores[seed] = best_logloss
    print(f"Seed [{seed}] Best Val Log-Loss: {best_logloss:.6f}")

print('\nSaved checkpoints:', best_ckpts)
print('Seed best scores:', seed_val_scores)




===== Seed 42 Training Start =====


Training:   0%|          | 0/32 [00:00<?, ?it/s]

Validation:   0%|          | 0/4 [00:00<?, ?it/s]

Seed [42] Epoch [1]
  - LR: 0.0002949
  - Train Loss: 0.3014
  - Val Log-Loss: 2.477224 | Val Acc: 0.5400


Training:   0%|          | 0/32 [00:00<?, ?it/s]

...

Validation:   0%|          | 0/4 [00:00<?, ?it/s]

Seed [777] Epoch [11]
  - LR: 0.0000061
  - Train Loss: 0.0018
  - Val Log-Loss: 0.246244 | Val Acc: 0.9200


Training:   0%|          | 0/32 [00:00<?, ?it/s]

Validation:   0%|          | 0/4 [00:00<?, ?it/s]

Seed [777] Epoch [12]
  - LR: 0.0000010
  - Train Loss: 0.0088
  - Val Log-Loss: 0.273775 | Val Acc: 0.8700
Seed [777] Best Val Log-Loss: 0.246244

Saved checkpoints: ['best_model_seed42.pt', 'best_model_seed3407.pt', 'best_model_seed777.pt']
Seed best scores: {42: np.float64(0.4994107531379413), 3407: np.float64(0.31857796782260106), 777: np.float64(0.24624442738048025)}


In [51]:
def infer_with_tta(model, views, use_hflip=True):
    probs_list = []

    # Original
    logits = model(views).view(-1)
    probs_list.append(torch.sigmoid(logits))

    # Horizontal flip TTA
    if use_hflip:
        flipped_views = [torch.flip(v, dims=[3]) for v in views]
        logits_flip = model(flipped_views).view(-1)
        probs_list.append(torch.sigmoid(logits_flip))

    return torch.stack(probs_list, dim=0).mean(dim=0)

# 학습된 시드별 best 모델 로드
ensemble_models = []
for seed in CFG['SEEDS']:
    ckpt_path = Path(f'best_model_seed{seed}.pt')
    if not ckpt_path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}")

    model = MultiViewResNet().to(device)
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.eval()
    ensemble_models.append(model)

print(f"Loaded {len(ensemble_models)} ensemble models")

all_probs = []

with torch.no_grad():
    for views in tqdm(test_loader, desc="Inference (Ensemble+TTA)"):
        views = [v.to(device) for v in views]

        per_model_probs = []
        for model in ensemble_models:
            probs = infer_with_tta(model, views, use_hflip=CFG['TTA_HFLIP'])
            per_model_probs.append(probs)

        batch_probs = torch.stack(per_model_probs, dim=0).mean(dim=0)
        all_probs.extend(batch_probs.cpu().numpy())

all_probs = np.array(all_probs, dtype=np.float64)
all_probs = np.clip(all_probs, 1e-4, 1 - 1e-4)

# 결과 저장 (컬럼 순서 중요)
submission = pd.DataFrame({
    'id': test_df['id'],
    'unstable_prob': all_probs,
    'stable_prob': 1.0 - all_probs
})

submission.to_csv('submission.csv', encoding='UTF-8-sig', index=False)
print("submission.csv 저장 완료.")



Loaded 3 ensemble models


Inference (Ensemble+TTA):   0%|          | 0/32 [00:00<?, ?it/s]

submission.csv 저장 완료.
